# MMHCL+ Training for Amazon Clothing Dataset (Local)
## MMHCL via Neighbor-Layer Hypergraph Contrastive Learning (Rev 5.2)

This notebook trains the **MMHCL+** model on the Amazon Clothing dataset on a local machine with NVIDIA RTX 5090 (32GB VRAM).

MMHCL+ (Rev 5.2) extends the original MMHCL (Multi-Modal Hypergraph Contrastive Learning) architecture with:
- **Neighbor-Layer CL** — positive pairs from adjacent GNN layers (NLGCL+ approach)
- **VICReg** on the user-user hypergraph branch (u2u) — D=4096 projector (was 1024 in Rev5.1)
- **Chunked InfoNCE** on the item-item hypergraph branch (i2i) with **delayed FAISS hard negative mining** (epoch 50+) and Adaptive Sample Weighting
- **Soft BYOL** cross-view alignment (hypergraph → bipartite embeddings)
- **Dirichlet Energy** regularisation to prevent over-smoothing
- **Hybrid Loss Balancer** — Uncertainty Weighting → GradNorm transition for 5-task balancing
- **CL Warmup Ramp** — linear 0→1 over 20 epochs after warmup (prevents CL Activation Shock)
- **SVD Spectral Augmentation** — filters popularity-dominated singular values from incidence matrix
- **Warmup epochs** (15) + exponential temperature annealing

Rev 5.2 changes vs Rev 5.1:
- Ego-Final Anchor **removed** (conflicts with neighbor-layer CL philosophy)
- Projector expanded: 64→1024→4096 (was 64→512→1024)
- Warmup increased to 15 epochs (was 5)
- FAISS hard negatives delayed until epoch 50 (was immediate)
- Hybrid Balancer replaces Uncertainty-only (5 tasks instead of 6)

## Configuration:
- **GPU**: NVIDIA RTX 5090 (32GB VRAM)
- **Batch Size**: 1024 (fit with original paper)
- **Max Epochs**: 250 (fit with original paper)
- **Warmup Epochs**: 15 (BPR-only phase to stabilise base embeddings)
- **W&B Project**: snips-local-mmhcl-plus-clothing
- **Training Script**: `codes/main_mmhcl_plus.py`

## Steps:
1. Verify environment and GPU
2. Setup W&B logging
3. Train MMHCL+ model with multiple seeds
4. Export results to Excel

## 1. Environment Setup & GPU Verification

In [ ]:
from __future__ import annotations

import os
import sys

import torch

# Get the project root directory (handle case where we might already be in codes dir)
current_dir: str = os.getcwd()
if current_dir.endswith("codes"):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

CODES_DIR: str = os.path.join(PROJECT_ROOT, "codes")
DATA_DIR: str = os.path.join(PROJECT_ROOT, "data")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Codes Directory: {CODES_DIR}")
print(f"Data Directory: {DATA_DIR}")

# Verify directories exist
assert os.path.exists(CODES_DIR), f"Codes directory not found: {CODES_DIR}"
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print("\n[OK] All directories verified")

# Verify GPU
print("\n" + "=" * 60)
print("GPU Information")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB"
    )
else:
    print("WARNING: No GPU detected! Training will be very slow.")

## 1.5 Install Dependencies (Run Once)

In [ ]:
# Install required dependencies (run this cell once)
# Configuration: PyTorch 2.0.1+cu118, NumPy 1.24.3, SciPy 1.10.1, scikit-learn 1.2.2
from __future__ import annotations

import os
import subprocess
import sys

print("=" * 60)
print("Installing Dependencies")
print("  - PyTorch 2.0.1+cu118")
print("  - NumPy 1.24.3")
print("  - SciPy 1.10.1")
print("  - scikit-learn 1.2.2")
print("=" * 60)

# Check if UV-managed Python (needs --break-system-packages)
pip_extra_args: list[str] = []
test_result: subprocess.CompletedProcess[str] = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--dry-run", "pip"],
    capture_output=True,
    text=True,
)
if "externally-managed" in test_result.stderr:
    pip_extra_args = ["--break-system-packages"]
    print("\n[INFO] UV-managed Python detected, using --break-system-packages")

# Step 1: Install from requirements.txt
requirements_path: str = os.path.join(PROJECT_ROOT, "requirements.txt")
if os.path.exists(requirements_path):
    print(f"\n1. Installing packages from: {requirements_path}")
    print("   This may take several minutes for PyTorch (~2.6GB)...")
    cmd: list[str] = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        requirements_path,
    ] + pip_extra_args
    result: subprocess.CompletedProcess[str] = subprocess.run(
        cmd, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"   [ERROR] Installation failed! Exit code: {result.returncode}")
        print("\n--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
        raise Exception("Failed to install requirements")
    print("   [OK] requirements.txt packages installed")
else:
    print(f"   [WARNING] requirements.txt not found at {requirements_path}")

# Step 2: Install additional packages
additional_packages: list[str] = ["wandb", "openpyxl", "pandas"]
print(f"\n2. Installing additional packages: {additional_packages}")
for package in additional_packages:
    print(f"   Installing {package}...")
    cmd: list[str] = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        package,
    ] + pip_extra_args
    subprocess.run(cmd, capture_output=True)
print("   [OK] Additional packages installed")

print("\n" + "=" * 60)
print("[OK] All packages installed successfully!")
print("=" * 60)
print("\n[IMPORTANT] Please RESTART THE KERNEL before running the next cells!")

## 2. Weights & Biases Setup

In [ ]:
from __future__ import annotations

from IPython.display import Markdown, display
import wandb

# ========== Login to Weights & Biases ==========
# W&B Configuration
wandb_project: str = "snips-local-mmhcl-plus-clothing"
wandb_entity: str = "baitapck51cc-uet"

# Your W&B API key
WANDB_API_KEY: str = (
    "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"
)

# Login to W&B
wandb.login(key=WANDB_API_KEY)

# Verify login
print("✓ W&B login successful")
print(f"✓ Entity: {wandb_entity}")
print(f"✓ Project: {wandb_project}")

# Construct W&B URL and display clickable link
wandb_url: str = f"https://wandb.ai/{wandb_entity}/{wandb_project}"
print()
display(Markdown(f"## 📊 [Click here to open Weights & Biases Dashboard]({wandb_url})"))
print(f"Direct Link: {wandb_url}")

## 2.5 GPU Memory Monitor (Optional)

Start a lightweight background monitor to print GPU VRAM usage while training.
Run the stop function after training completes.

In [ ]:
from __future__ import annotations

import csv
import os
import threading
import time
from typing import Optional

import torch

# GPU monitor controls
GPU_MONITOR_RUNNING: bool = False
GPU_MONITOR_THREAD: threading.Thread | None = None


def _resolve_monitor_root() -> str:
    current_dir: str = os.getcwd()
    if current_dir.endswith("codes"):
        return os.path.dirname(current_dir)
    return current_dir


def _gpu_monitor(interval_seconds: int, csv_path: str, print_live: bool) -> None:
    global GPU_MONITOR_RUNNING
    if not torch.cuda.is_available():
        print("[WARNING] CUDA not available. GPU monitor will not run.")
        GPU_MONITOR_RUNNING = False
        return

    total_mem: int = torch.cuda.get_device_properties(0).total_memory
    start_time: float = time.time()

    file_exists: bool = os.path.exists(csv_path)
    with open(csv_path, "a", newline="", encoding="utf-8") as csv_file:
        writer: csv.writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow(
                [
                    "relative_time_s",
                    "allocated_gb",
                    "reserved_gb",
                    "allocated_pct",
                    "reserved_pct",
                ]
            )

        print(f"[INFO] GPU monitor started. Logging to: {csv_path}")
        while GPU_MONITOR_RUNNING:
            allocated: int = torch.cuda.memory_allocated(0)
            reserved: int = torch.cuda.memory_reserved(0)
            allocated_pct: float = (allocated / total_mem) * 100
            reserved_pct: float = (reserved / total_mem) * 100
            rel_time: float = time.time() - start_time

            writer.writerow(
                [
                    f"{rel_time:.2f}",
                    f"{allocated / 1024**3:.4f}",
                    f"{reserved / 1024**3:.4f}",
                    f"{allocated_pct:.2f}",
                    f"{reserved_pct:.2f}",
                ]
            )
            csv_file.flush()

            if print_live:
                print(
                    f"[GPU] allocated={allocated / 1024**3:.2f} GB ({allocated_pct:.1f}%), "
                    f"reserved={reserved / 1024**3:.2f} GB ({reserved_pct:.1f}%)"
                )

            time.sleep(interval_seconds)

    print("[INFO] GPU monitor stopped.")


def start_gpu_monitor(
    interval_seconds: int = 10,
    csv_filename: str = "gpu_memory_monitor.csv",
    print_live: bool = False,
) -> None:
    """Start background GPU monitor. Logs to CSV every N seconds."""
    global GPU_MONITOR_RUNNING, GPU_MONITOR_THREAD
    if GPU_MONITOR_RUNNING:
        print("[INFO] GPU monitor already running.")
        return

    monitor_root: str = _resolve_monitor_root()
    csv_path: str = os.path.join(monitor_root, csv_filename)

    GPU_MONITOR_RUNNING = True
    GPU_MONITOR_THREAD = threading.Thread(
        target=_gpu_monitor, args=(interval_seconds, csv_path, print_live), daemon=True
    )
    GPU_MONITOR_THREAD.start()


def stop_gpu_monitor() -> None:
    """Stop background GPU monitor."""
    global GPU_MONITOR_RUNNING
    GPU_MONITOR_RUNNING = False


# Start monitor with 15s interval (adjust as needed)
# Set print_live=True if you want console output too
start_gpu_monitor(
    interval_seconds=15, csv_filename="gpu_memory_monitor.csv", print_live=False
)

# Call stop_gpu_monitor() after training completes.

## 3. Multi-Seed Training (MMHCL+ Rev 5.2)

Train the **MMHCL+ (Neighbor-Layer Hypergraph CL)** model with multiple random seeds.

### Base Hyperparameters (same as original MMHCL):
- **Batch Size**: 1024 (fit with original paper, NVIDIA RTX 5090 32GB)
- **Max Epochs**: 250 (fit with original paper)
- **Learning Rate**: 0.0001
- **Early Stopping**: patience=30, min_epochs=75
- **ReduceLROnPlateau**: enabled (factor=0.5, patience=3)

### MMHCL+ Rev 5.2 Changes (vs Rev 5.1):
- **Warmup Epochs**: 15 — extended BPR-only phase (was 5 in Rev5.1)
- **CL Ramp**: linear 0→1 over 20 epochs after warmup (prevents CL Activation Shock)
- **Temperature Schedule**: tau anneals from 0.5 → 0.05 with gamma=0.99
- **VICReg** sim=25.0, var=25.0, cov=1.0 (u2u branch, projector 64→1024→4096)
- **Chunked InfoNCE** chunk_size=512 (i2i branch) with 10 FAISS hard negatives (delayed until epoch 50)
- **Ego-Final Anchor**: REMOVED (conflicts with neighbor-layer CL philosophy)
- **Soft BYOL** alignment weight=1.0 (hypergraph → bipartite stop-grad target)
- **Dirichlet Energy** weight=1.0 (over-smoothing regularisation)
- **Hybrid Balancer**: Uncertainty Weighting → GradNorm transition (5-task balancing)
- **SVD Filtering**: top-10 singular values zeroed (popularity debiasing)
- **WEMAManager**: initialised from image_feat.npy + text_feat.npy for item-side ASW

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import random
import re
import subprocess
import sys
import time
from typing import Any, Optional

import numpy as np

# Define project directories (handle case where we might already be in codes dir)
current_dir: str = os.getcwd()
if current_dir.endswith("codes"):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT: str = current_dir

CODES_DIR: str = os.path.join(PROJECT_ROOT, "codes")
DATA_DIR: str = os.path.join(PROJECT_ROOT, "data")

# Change to codes directory (only if not already there)
if not os.getcwd().endswith("codes"):
    os.chdir(CODES_DIR)
if CODES_DIR not in sys.path:
    sys.path.insert(0, CODES_DIR)

# Get the Python executable path (same as Jupyter kernel)
PYTHON_EXE: str = sys.executable

print(f"Project Root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")
print(f"Python executable: {PYTHON_EXE}")

print("\n" + "=" * 80)
print("MMHCL+ Rev5.2 MULTI-SEED TRAINING: Running 3 experiments with different random seeds")
print("=" * 80)

# Generate truly random seeds based on current time
base_seed: int = int(time.time_ns() % (2**31))
random.seed(base_seed)

# Generate 3 different random seeds
n_runs: int = 3
seeds: list[int] = [random.randint(1, 2**31 - 1) for _ in range(n_runs)]

print(f"\nGenerated {n_runs} random seeds based on current time:")
print(f"Base timestamp seed: {base_seed}")
print(f"Seeds for experiments: {seeds}")
print("(These seeds will be saved in the summary file for reproducibility)")

# W&B Configuration
wandb_project: str = "snips-local-mmhcl-plus-clothing"
wandb_entity: str = "baitapck51cc-uet"

# Base hyperparameters (same as original paper)
dataset: str = "Clothing"
gpu_id: int = 0
epoch: int = 250  # Maximum epochs (fit with original paper)
verbose: int = 5

# Batch size 1024 (fit with original paper, RTX 5090 32GB VRAM)
batch_size: int = 1024
lr: float = 0.0001
regs: float = 1e-3
embed_size: int = 64
topk: int = 5
core: int = 5
User_layers: int = 3
Item_layers: int = 2
user_loss_ratio: float = 0.03
item_loss_ratio: float = 0.07
temperature: float = (
    0.6  # kept for path naming; actual tau uses tau_max/tau_min/tau_gamma
)

# ── MMHCL+ Rev5.2 additions ───────────────────────────────────────────────────

# Warmup + temperature annealing
warmup_epochs: int = 15  # BPR-only stabilisation before CL kicks in (Rev5.2: 15)
tau_max: float = 0.5  # initial temperature for i2i InfoNCE
tau_min: float = 0.05  # floor temperature
tau_gamma: float = 0.99  # per-epoch decay factor

# VICReg settings (replaces Barlow Twins in Rev5.1)
vicreg_sim_weight: float = 25.0  # invariance (MSE) coefficient
vicreg_var_weight: float = 25.0  # variance (hinge) coefficient
vicreg_cov_weight: float = 1.0   # covariance (off-diagonal) coefficient

# Loss function settings
info_nce_chunk_size: int = 512  # memory-safe InfoNCE chunk size
num_tasks: int = 5  # Rev5.2: 5 tasks (BPR, u2u, i2i, align, dir). Ego-final removed.

# Per-task initial weights
bpr_weight: float = 1.0
u2u_weight: float = 1.0
i2i_weight: float = 1.0
align_weight: float = 1.0
dirichlet_weight: float = 1.0
ego_final_weight: float = 0.0  # [DEPRECATED Rev5.2] Ego-final anchor removed

# Expanded projector dimensions (u2u branch)
# Rev5.2: 64 -> 1024 -> 4096 (was 64 -> 512 -> 1024 in Rev5.1)
projector_hidden_dim: int = 1024
projector_out_dim: int = 4096

# SVD Spectral Augmentation (Rev5.1)
use_svd_filtering: int = 1  # 1=enable SVD filtering
svd_top_k: int = 10  # zero top-10 singular values (popularity-dominated)

# FAISS Hard Negative Mining (Rev5.1)
n_hard_neg: int = 10  # hard negatives per query
hard_neg_pool_k: int = 64  # ANN pool size
hard_neg_weight: float = 0.5  # weight in InfoNCE denominator

# WEMAManager (Adaptive Sample Weighting — item AND user sides)
w_ema_alpha: float = 0.9
w_ema_update_interval: int = 5

# Soft Topology-Aware Purification
soft_purification_percentile: float = 0.8
soft_purification_temp: float = 0.2

# EMA Teacher Network
ema_momentum: float = 0.99  # teacher momentum (BYOL default)

# Profiling-Guided Activation Checkpointing
checkpoint_threshold: int = -1  # -1 = disable

# Rev5.2: CL Ramp, Delayed FAISS, Hybrid Balancer
cl_ramp_epochs: int = 20  # linear ramp CL losses 0→1 after warmup
delay_hard_negs_epoch: int = 50  # FAISS hard negatives activate after this epoch
use_hybrid_balancer: int = 1  # 1=Hybrid (Uncertainty→GradNorm); 0=Uncertainty-only
balancer_transition_epoch: int = 40  # start transitioning to GradNorm
balancer_blend_epochs: int = 20  # epochs to blend Uncertainty→GradNorm

# ─────────────────────────────────────────────────────────────────────────────

# Early stopping configuration
early_stopping_patience: int = 30
early_stopping_min_epochs: int = 75  # Adjusted proportionally for 250 max epochs
early_stopping_min_delta: float = 0.0001
early_stopping_monitor: str = "val_recall@20"
early_stopping_mode: str = "max"
early_stopping_restore_best: int = 1
adaptive_patience: int = 0  # Disabled

# ReduceLROnPlateau scheduler
use_reduce_lr: int = 1
reduce_lr_factor: float = 0.5
reduce_lr_patience: int = 3
reduce_lr_min: float = 1e-6

# Storage for results
all_results: list[dict[str, Any]] = []

print(f"\n{'=' * 80}")
print("Training Configuration (MMHCL+ Rev5.2):")
print(f"{'=' * 80}")
print(f"  Dataset:          {dataset}")
print(f"  GPU ID:           {gpu_id}")
print(f"  Max Epochs:       {epoch}")
print(f"  Batch Size:       {batch_size}")
print(f"  Learning Rate:    {lr}")
print(f"  Warmup Epochs:    {warmup_epochs}")
print(f"  CL Ramp Epochs:   {cl_ramp_epochs}")
print(f"  Tau schedule:     {tau_max} -> {tau_min} (gamma={tau_gamma})")
print(f"  VICReg weights:   sim={vicreg_sim_weight}, var={vicreg_var_weight}, cov={vicreg_cov_weight}")
print(f"  Projector:        {embed_size} -> {projector_hidden_dim} -> {projector_out_dim}")
print(f"  Hard negatives:   n={n_hard_neg}, pool_k={hard_neg_pool_k}, weight={hard_neg_weight}, delay_epoch={delay_hard_negs_epoch}")
print(f"  SVD filtering:    {'enabled' if use_svd_filtering else 'disabled'} (top_k={svd_top_k})")
print(f"  Tasks:            {num_tasks} (BPR, u2u, i2i, align, dir)")
print(f"  Balancer:         {'Hybrid (Uncertainty→GradNorm)' if use_hybrid_balancer else 'Uncertainty-only'}")
print(
    f"  Early Stopping:   patience={early_stopping_patience}, min_epochs={early_stopping_min_epochs}"
)
print(f"  W&B Project:      {wandb_project}")
print("  Training Script:  main_mmhcl_plus.py")

# Run training for each seed
for run_idx, seed in enumerate(seeds, 1):
    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx}/{n_runs}: Training MMHCL+ Rev5.2 with seed={seed}")
    print(f"{'=' * 80}")

    # Build training command (use PYTHON_EXE to ensure same Python as Jupyter kernel)
    cmd: list[str] = [
        PYTHON_EXE,
        "main_mmhcl_plus.py",
        # ── Base MMHCL args ──────────────────────────────────────────────────
        "--dataset",
        dataset,
        "--gpu_id",
        str(gpu_id),
        "--seed",
        str(seed),
        "--epoch",
        str(epoch),
        "--verbose",
        str(verbose),
        "--batch_size",
        str(batch_size),
        "--lr",
        str(lr),
        "--regs",
        str(regs),
        "--embed_size",
        str(embed_size),
        "--topk",
        str(topk),
        "--core",
        str(core),
        "--User_layers",
        str(User_layers),
        "--Item_layers",
        str(Item_layers),
        "--user_loss_ratio",
        str(user_loss_ratio),
        "--item_loss_ratio",
        str(item_loss_ratio),
        "--temperature",
        str(temperature),
        "--early_stopping_patience",
        str(early_stopping_patience),
        "--early_stopping_min_epochs",
        str(early_stopping_min_epochs),
        "--early_stopping_min_delta",
        str(early_stopping_min_delta),
        "--early_stopping_monitor",
        str(early_stopping_monitor),
        "--early_stopping_mode",
        str(early_stopping_mode),
        "--early_stopping_restore_best",
        str(early_stopping_restore_best),
        "--adaptive_patience",
        str(adaptive_patience),
        "--use_reduce_lr",
        str(use_reduce_lr),
        "--reduce_lr_factor",
        str(reduce_lr_factor),
        "--reduce_lr_patience",
        str(reduce_lr_patience),
        "--reduce_lr_min",
        str(reduce_lr_min),
        # ── MMHCL+ Rev5.2 additions ─────────────────────────────────────────
        "--warmup_epochs",
        str(warmup_epochs),
        "--tau_max",
        str(tau_max),
        "--tau_min",
        str(tau_min),
        "--tau_gamma",
        str(tau_gamma),
        "--vicreg_sim_weight",
        str(vicreg_sim_weight),
        "--vicreg_var_weight",
        str(vicreg_var_weight),
        "--vicreg_cov_weight",
        str(vicreg_cov_weight),
        "--info_nce_chunk_size",
        str(info_nce_chunk_size),
        "--num_tasks",
        str(num_tasks),
        "--bpr_weight",
        str(bpr_weight),
        "--u2u_weight",
        str(u2u_weight),
        "--i2i_weight",
        str(i2i_weight),
        "--align_weight",
        str(align_weight),
        "--dirichlet_weight",
        str(dirichlet_weight),
        "--ego_final_weight",
        str(ego_final_weight),
        "--projector_hidden_dim",
        str(projector_hidden_dim),
        "--projector_out_dim",
        str(projector_out_dim),
        "--use_svd_filtering",
        str(use_svd_filtering),
        "--svd_top_k",
        str(svd_top_k),
        "--n_hard_neg",
        str(n_hard_neg),
        "--hard_neg_pool_k",
        str(hard_neg_pool_k),
        "--hard_neg_weight",
        str(hard_neg_weight),
        "--w_ema_alpha",
        str(w_ema_alpha),
        "--w_ema_update_interval",
        str(w_ema_update_interval),
        "--soft_purification_percentile",
        str(soft_purification_percentile),
        "--soft_purification_temp",
        str(soft_purification_temp),
        "--ema_momentum",
        str(ema_momentum),
        "--checkpoint_threshold",
        str(checkpoint_threshold),
        # ── Rev5.2-specific args ─────────────────────────────────────────────
        "--cl_ramp_epochs",
        str(cl_ramp_epochs),
        "--delay_hard_negs_epoch",
        str(delay_hard_negs_epoch),
        "--use_hybrid_balancer",
        str(use_hybrid_balancer),
        "--balancer_transition_epoch",
        str(balancer_transition_epoch),
        "--balancer_blend_epochs",
        str(balancer_blend_epochs),
        # ── W&B ──────────────────────────────────────────────────────────────
        "--use_wandb",
        "1",
        "--wandb_project",
        wandb_project,
        "--wandb_entity",
        wandb_entity,
        "--wandb_run_name",
        f"mmhcl_plus_v52_clothing_seed_{seed}",
    ]

    print(f"Command: {' '.join(cmd)}")
    print(f"Current directory: {os.getcwd()}\n")

    # Run training with combined stdout/stderr for real-time output
    # Use PYTHONIOENCODING to avoid Windows encoding issues
    env: dict[str, str] = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUNBUFFERED"] = "1"

    result: subprocess.CompletedProcess[str] = subprocess.run(
        cmd, capture_output=True, text=True, env=env, encoding="utf-8", errors="replace"
    )

    # Always print stdout if there's any output
    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print(
            f"\n[WARNING] Training with seed={seed} exited with code {result.returncode}"
        )
        # Show error output to help debug
        if result.stderr:
            print("\n[ERROR OUTPUT]:")
            print(result.stderr)
    else:
        print(f"\n[OK] Training with seed={seed} completed successfully")

    # Extract results from log file
    path_name: str = (
        f"uu_ii={User_layers}_{Item_layers}_{user_loss_ratio}_{item_loss_ratio}_topk={topk}_t={temperature}_regs={regs}_dim={embed_size}_seed={seed}_"
    )
    log_file: str = f"../{dataset}/{path_name}/{path_name}.txt"

    if os.path.exists(log_file):
        with open(log_file) as f:
            log_content: str = f.read()

        # Extract BEST test metrics
        recall_match: re.Match[str] | None = re.search(
            r"BEST_Test_Recall@20:\s+([\d.]+)", log_content
        )
        precision_match: re.Match[str] | None = re.search(
            r"BEST_Test_Precision@20:\s+([\d.]+)", log_content
        )
        ndcg_match: re.Match[str] | None = re.search(
            r"BEST_Test_NDCG@20:\s+([\d.]+)", log_content
        )

        # Fallback to old format
        if not recall_match:
            recall_match = re.search(r"Test_Recall@20:\s+([\d.]+)", log_content)
        if not precision_match:
            precision_match = re.search(r"Test_Precision@20:\s+([\d.]+)", log_content)
        if not ndcg_match:
            ndcg_match = re.search(r"Test_NDCG@20:\s+([\d.]+)", log_content)

        if recall_match and precision_match and ndcg_match:
            run_result: dict[str, Any] = {
                "seed": seed,
                "recall@20": float(recall_match.group(1)),
                "precision@20": float(precision_match.group(1)),
                "ndcg@20": float(ndcg_match.group(1)),
                "log_file": log_file,
            }
            all_results.append(run_result)
            print(
                f"  Extracted metrics: Recall@20={run_result['recall@20']:.6f}, "
                f"Precision@20={run_result['precision@20']:.6f}, "
                f"NDCG@20={run_result['ndcg@20']:.6f}"
            )
        else:
            print(f"  Could not extract metrics from log file: {log_file}")
    else:
        print(f"  Log file not found: {log_file}")

print(f"\n{'=' * 80}")
print("ALL TRAINING RUNS COMPLETED")
print(f"{'=' * 80}")

## 4. Results Summary & Statistics

In [ ]:
from __future__ import annotations

import json
from typing import Any

import numpy as np

# Compute and display statistics
if len(all_results) > 0:
    print(f"\n{'=' * 80}")
    print(f"RESULTS SUMMARY (Mean ± Std across {len(all_results)} runs)")
    print(f"{'=' * 80}")

    metrics: list[str] = ["recall@20", "precision@20", "ndcg@20"]
    stats: dict[str, dict[str, Any]] = {}

    for metric in metrics:
        values: list[float] = [r[metric] for r in all_results]
        mean_val: np.floating = np.mean(values)
        std_val: np.floating = np.std(values)
        stats[metric] = {"mean": mean_val, "std": std_val, "values": values}

        print(f"\n{metric.upper()}:")
        print(f"  Mean: {mean_val:.6f}")
        print(f"  Std:  {std_val:.6f}")
        print(f"  Mean ± Std: {mean_val:.6f} ± {std_val:.6f}")
        print(f"  Individual values: {[f'{v:.6f}' for v in values]}")

    # Compare with reference values (MMHCL baseline on Amazon Clothing).
    print(f"\n{'=' * 80}")
    print("COMPARISON WITH REFERENCE BASELINE (MMHCL - Amazon Clothing)")
    print("NOTE: MMHCL paper does not report Amazon Clothing results.")
    print("      Reference: Original MMHCL (ACM TOMM 2025) - Amazon Clothing (5-core).")
    print(f"{'=' * 80}")
    paper_values: dict[str, float] = {
        "recall@20": 0.0907,
        "precision@20": 0.0051,
        "ndcg@20": 0.0443,
    }
    for metric in metrics:
        our_mean: Any = stats[metric]["mean"]
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100
        status: str = "✓ OK" if abs(diff_pct) < 10 else "⚠️ CHECK"
        print(
            f"  {metric.upper()}: Ours={our_mean:.4f} vs Paper={paper_val:.4f} ({diff_pct:+.1f}%) [{status}]"
        )

    # Save summary to file
    summary_file: str = f"../{dataset}/multi_seed_summary.json"
    summary_data: dict[str, Any] = {
        "n_runs": len(all_results),
        "seeds": [r["seed"] for r in all_results],
        "config": {
            "batch_size": batch_size,
            "max_epochs": epoch,
            "lr": lr,
            "early_stopping_patience": early_stopping_patience,
            "early_stopping_min_epochs": early_stopping_min_epochs,
            "wandb_project": wandb_project,
        },
        "statistics": {
            k: {"mean": float(v["mean"]), "std": float(v["std"])}
            for k, v in stats.items()
        },
        "individual_results": all_results,
    }

    with open(summary_file, "w") as f:
        json.dump(summary_data, f, indent=2)

    print(f"\n✓ Summary saved to: {summary_file}")

    # Human-readable summary
    summary_txt_file: str = f"../{dataset}/multi_seed_summary.txt"
    with open(summary_txt_file, "w") as f:
        f.write("=" * 80 + "\n")
        f.write(f"MULTI-SEED TRAINING RESULTS ({len(all_results)} runs)\n")
        f.write(
            f"Configuration: batch_size={batch_size}, max_epochs={epoch}, lr={lr}\n"
        )
        f.write("=" * 80 + "\n\n")
        f.write(f"Seeds used: {seeds[: len(all_results)]}\n\n")

        for metric in metrics:
            f.write(f"{metric.upper()}:\n")
            f.write(
                f"  Mean ± Std: {stats[metric]['mean']:.6f} ± {stats[metric]['std']:.6f}\n"
            )
            f.write(
                f"  Individual: {[f'{v:.6f}' for v in stats[metric]['values']]}\n\n"
            )

        f.write("\nIndividual Run Results:\n")
        f.write("-" * 80 + "\n")
        for r in all_results:
            f.write(
                f"Seed {r['seed']:10d}: Recall@20={r['recall@20']:.6f}, "
                f"Precision@20={r['precision@20']:.6f}, "
                f"NDCG@20={r['ndcg@20']:.6f}\n"
            )

    print(f"✓ Human-readable summary saved to: {summary_txt_file}")
else:
    print("\n⚠️ WARNING: No results were extracted from any training run!")
    print("Please check the log files manually.")

## 5. Export Results to Excel

In [ ]:
from __future__ import annotations

from typing import Any

from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
import pandas as pd

if len(all_results) > 0:
    print("=" * 80)
    print("EXPORTING RESULTS TO EXCEL")
    print("=" * 80)

    excel_file: str = f"../{dataset}/MMHCL_{dataset}_Results_Local.xlsx"

    # Prepare Individual Results data
    individual_data: list[dict[str, Any]] = []
    for i, r in enumerate(all_results, 1):
        row: dict[str, Any] = {
            "Run": i,
            "Seed": r["seed"],
            "Recall@20": r["recall@20"],
            "Precision@20": r["precision@20"],
            "NDCG@20": r["ndcg@20"],
        }
        individual_data.append(row)

    df_individual: pd.DataFrame = pd.DataFrame(individual_data)

    # Prepare Summary Statistics data
    summary_data_list: list[dict[str, Any]] = []
    for metric in metrics:
        metric_name: str = metric.upper()
        summary_data_list.append(
            {
                "Metric": metric_name,
                "Mean": stats[metric]["mean"],
                "Std": stats[metric]["std"],
                "Mean ± Std": f"{stats[metric]['mean']:.6f} ± {stats[metric]['std']:.6f}",
            }
        )

    df_summary: pd.DataFrame = pd.DataFrame(summary_data_list)

    # Paper comparison data
    comparison_data: list[dict[str, Any]] = []
    for metric in metrics:
        our_mean: Any = stats[metric]["mean"]
        paper_val: float = paper_values[metric]
        diff_pct: float = ((our_mean - paper_val) / paper_val) * 100
        comparison_data.append(
            {
                "Metric": metric.upper(),
                "Our Result": our_mean,
                "Paper Result": paper_val,
                "Difference (%)": diff_pct,
            }
        )

    df_comparison: pd.DataFrame = pd.DataFrame(comparison_data)

    # Write to Excel with formatting
    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        df_individual.to_excel(writer, sheet_name="Individual Results", index=False)
        df_summary.to_excel(writer, sheet_name="Summary Statistics", index=False)
        df_comparison.to_excel(writer, sheet_name="Paper Comparison", index=False)

        # Format sheets
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            # Auto-adjust column widths
            for col in ws.columns:
                max_length: int = 0
                column: str = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                ws.column_dimensions[column].width = max_length + 2

            # Bold header row
            for cell in ws[1]:
                cell.font = Font(bold=True)
                cell.fill = PatternFill(
                    start_color="E0E0E0", end_color="E0E0E0", fill_type="solid"
                )

    print(f"\n✓ Results exported to: {excel_file}")
    print(f"  - Sheet 1: Individual Results ({len(all_results)} runs)")
    print("  - Sheet 2: Summary Statistics")
    print("  - Sheet 3: Paper Comparison")
else:
    print("No results to export.")

## 6. Check W&B Run Status (Optional)

In [ ]:
from __future__ import annotations

import wandb

print(f"Checking runs for {wandb_entity}/{wandb_project}...")

try:
    api: wandb.Api = wandb.Api()
    runs: wandb.apis.public.Runs = api.runs(f"{wandb_entity}/{wandb_project}")

    if len(runs) == 0:
        print("No runs found in this project yet.")
    else:
        print(f"\nFound {len(runs)} runs. Latest 5 status:")
        print("-" * 60)
        for run in runs[:5]:
            print(f"Run Name: {run.name}")
            print(f"Status:   {run.state}")
            print(f"Created:  {run.created_at}")
            if "val/recall" in run.summary:
                print(f"Val Recall: {run.summary['val/recall']:.4f}")
            if "best_recall" in run.summary:
                print(f"Best Recall: {run.summary['best_recall']:.4f}")
            print("-" * 60)

except Exception as e:
    print(f"Error fetching run status: {e}")
    print("Make sure W&B is logged in correctly.")

## Troubleshooting

### Common Issues:

1. **CUDA Out of Memory**: Reduce batch size further (256 or 128). Rev5.2 uses a larger VICReg
   projector (D=4096) than Rev5.1 (D=1024), so monitor VRAM usage accordingly.
   ```python
   batch_size = 256  # or 128
   ```

2. **W&B login issues**: Run `wandb login` in terminal first

3. **Module not found**: Install dependencies
   ```bash
   pip install torch wandb numpy pandas openpyxl scipy faiss-cpu
   ```

4. **Data not found**: Ensure all preprocessed files are present in `data/Clothing/`:
   - `data/Clothing/5-core/train.json`
   - `data/Clothing/5-core/val.json`
   - `data/Clothing/5-core/test.json`
   - `data/Clothing/image_feat.npy`  (CLIP ViT-B/32, 512-d, 7050 items)
   - `data/Clothing/text_feat.npy`   (CLIP ViT-B/32, 512-d, 7050 items)

5. **KeyError / shape mismatch on image/text features**: Re-run
   `reextract_image_features.py` and `verify_and_fix_text_features.py` from project root.

6. **FAISS not found**: Hard negative mining falls back to torch-based ANN if faiss is unavailable.
   Install faiss for best performance: `pip install faiss-cpu` (or `faiss-gpu` for GPU support).

### Rev5.2 Architecture Changes (vs Rev5.1):
| Component | Rev5.1 | Rev5.2 |
|-----------|--------|--------|
| Tasks | 6 (incl. ego-final) | 5 (ego-final removed) |
| u2u Projector | 64→512→1024 | 64→1024→4096 |
| Warmup Epochs | 5 | 15 |
| CL Ramp | None (immediate) | Linear 0→1 over 20 epochs |
| Hard Negatives | Immediate | Delayed to epoch 50 |
| Loss Balancer | Uncertainty only | Hybrid (Uncertainty→GradNorm) |
| Ego-Final Anchor | Active (weight=1.0) | Removed (weight=0.0) |

### Reference Results (Amazon Clothing):
The original MMHCL paper does **not** report results on Amazon Clothing.
BM3 (WWW '23) Table 3 reference baseline (per-user 8:1:1 random split):
- Recall@20:    0.0883
- Precision@20: 0.0044
- NDCG@20:      0.0383

Both MMHCL and BM3 use per-user 8:1:1 random split, ensuring all 19,445 users are evaluated.

## 7. MMHCL+ Q1-style Ablation Study (Rev 5.2)

This section drives the component-level, hyperparameter, and balancer ablations defined in
`mmhcl_plus_ablation_guide_full_translation.tex` and implemented in
`codes/mmhcl_plus/ablation/ablation_config.py`.

For every registered variant we:

1. Launch `main_mmhcl_plus.py --ablation_variant <name> --seed <seed>` over
   the configured list of seeds.
2. Parse `BEST_Test_Recall@20 / Precision@20 / NDCG@20` from the log file of
   each run.
3. Build a long-format dataframe (`variant × seed × metric`) and compute the
   marginal delta `Δ = metric(variant) − metric(A0_full)` averaged across
   seeds.
4. Run a paired t-test and a Wilcoxon signed-rank test against `A0_full`
   for statistical significance.
5. Export the mean, std, delta, and p-values to
   `ablation_results.{csv,xlsx}`.

The driver is CLI-compatible with the `ABLATION=A4_no_ramp python ...` idiom
described in the guide: setting `ABLATION_VARIANTS = ["A4_no_ramp"]` runs
only that variant.

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np

# -- Path plumbing (same pattern as Cell 10) -----------------------------
current_dir: str = os.getcwd()
if current_dir.endswith("codes"):
    PROJECT_ROOT: str = os.path.dirname(current_dir)
else:
    PROJECT_ROOT = current_dir
CODES_DIR: str = os.path.join(PROJECT_ROOT, "codes")
if not os.getcwd().endswith("codes"):
    os.chdir(CODES_DIR)
if CODES_DIR not in sys.path:
    sys.path.insert(0, CODES_DIR)
PYTHON_EXE: str = sys.executable

# -- Import the central registry -----------------------------------------
from mmhcl_plus.ablation import REGISTRY, get as get_variant, summarise

print(summarise())
print()

# -- Pick which variants + seeds to run ----------------------------------
# Override via environment variable: ABLATION=A4_no_ramp,A7_ego_final python ...
_env_variants: str = os.environ.get("ABLATION", "").strip()
if _env_variants:
    ABLATION_VARIANTS: list[str] = [v.strip() for v in _env_variants.split(",") if v.strip()]
else:
    ABLATION_VARIANTS = [
        "A0_full",
        "A1_no_nlcl",
        "A2_no_svd",
        "A3_small_proj",
        "A4_no_ramp",
        "A5_no_delay",
        "A6_no_dirichlet",
        "A7_ego_final",
        "A8_no_cross",
        "B1_g1",
        "B2_g2",
        "B3_g3",
        "C1_uncertainty",
        "C2_gradnorm",
        "C3_fixed",
    ]

# Deterministic seed list: keep 3 for Q1-grade paired statistics.
random.seed(int(time.time_ns() % (2**31)))
N_SEEDS: int = int(os.environ.get("ABLATION_N_SEEDS", "3"))
ABLATION_SEEDS: list[int] = [random.randint(1, 2**31 - 1) for _ in range(N_SEEDS)]

# -- Optional epoch cap (shorter runs while validating the pipeline) -----
# Set ABLATION_EPOCHS to a small value (e.g. 30) for a smoke run; leave
# empty to inherit the notebook's max-epoch default of 250.
_epoch_env: str = os.environ.get("ABLATION_EPOCHS", "").strip()
ABLATION_EPOCHS: int = int(_epoch_env) if _epoch_env else 250

print("=" * 80)
print(f"Ablation plan: {len(ABLATION_VARIANTS)} variants x {N_SEEDS} seeds")
print(f"Variants: {ABLATION_VARIANTS}")
print(f"Seeds:    {ABLATION_SEEDS}")
print(f"Max epochs per run: {ABLATION_EPOCHS}")
print("=" * 80)

# -- Storage for raw results ---------------------------------------------
ABLATION_RESULTS: list[dict[str, Any]] = []

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
import time
from typing import Any

# Base hyperparameters (must mirror Cell 10's Rev5.2 defaults).
# These are inherited from Cell 10's namespace; re-declare here so the
# ablation driver remains self-contained.
DATASET: str = "Clothing"
GPU_ID: int = 0
VERBOSE: int = 5
BATCH_SIZE: int = 1024
LR: float = 0.0001
REGS: float = 1e-3
EMBED_SIZE: int = 64
TOPK: int = 5
CORE: int = 5
USER_LAYERS: int = 3
ITEM_LAYERS: int = 2
USER_LOSS_RATIO: float = 0.03
ITEM_LOSS_RATIO: float = 0.07
TEMPERATURE: float = 0.6
WANDB_PROJECT_ABL: str = "snips-local-mmhcl-plus-clothing-ablation"
WANDB_ENTITY_ABL: str = "baitapck51cc-uet"
USE_WANDB_ABL: int = 1


def _build_run_cmd(variant_name: str, seed: int) -> list[str]:
    """Compose the CLI invocation for one (variant, seed) run.

    Only `--ablation_variant` is required: the registry override inside
    `main_mmhcl_plus.py` fills in the rest (projector dims, balancer
    mode, toggles, g_layers, ...).
    """
    return [
        PYTHON_EXE,
        "main_mmhcl_plus.py",
        # Dataset / training schedule
        "--dataset", DATASET,
        "--gpu_id", str(GPU_ID),
        "--seed", str(seed),
        "--epoch", str(ABLATION_EPOCHS),
        "--verbose", str(VERBOSE),
        "--batch_size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--regs", str(REGS),
        "--embed_size", str(EMBED_SIZE),
        "--topk", str(TOPK),
        "--core", str(CORE),
        "--User_layers", str(USER_LAYERS),
        "--Item_layers", str(ITEM_LAYERS),
        "--user_loss_ratio", str(USER_LOSS_RATIO),
        "--item_loss_ratio", str(ITEM_LOSS_RATIO),
        "--temperature", str(TEMPERATURE),
        "--early_stopping_patience", "30",
        "--early_stopping_min_epochs", str(min(75, ABLATION_EPOCHS // 2)),
        "--early_stopping_min_delta", "0.0001",
        "--early_stopping_monitor", "val_recall@20",
        "--early_stopping_mode", "max",
        "--early_stopping_restore_best", "1",
        "--use_reduce_lr", "1",
        "--reduce_lr_factor", "0.5",
        "--reduce_lr_patience", "3",
        "--reduce_lr_min", "1e-6",
        # Ablation override -- everything else is filled in by the registry.
        "--ablation_variant", variant_name,
        # W&B
        "--use_wandb", str(USE_WANDB_ABL),
        "--wandb_project", WANDB_PROJECT_ABL,
        "--wandb_entity", WANDB_ENTITY_ABL,
        "--wandb_run_name", f"abl_{variant_name}_seed_{seed}",
    ]


_BEST_PATTERNS: dict[str, re.Pattern[str]] = {
    "recall@20": re.compile(r"BEST_Test_Recall@20:\s+([\d.]+)"),
    "precision@20": re.compile(r"BEST_Test_Precision@20:\s+([\d.]+)"),
    "ndcg@20": re.compile(r"BEST_Test_NDCG@20:\s+([\d.]+)"),
}
_FALLBACK_PATTERNS: dict[str, re.Pattern[str]] = {
    "recall@20": re.compile(r"Test_Recall@20:\s+([\d.]+)"),
    "precision@20": re.compile(r"Test_Precision@20:\s+([\d.]+)"),
    "ndcg@20": re.compile(r"Test_NDCG@20:\s+([\d.]+)"),
}


def _parse_log(variant_name: str, seed: int) -> dict[str, float] | None:
    """Parse the per-run log file produced by build_experiment_paths."""
    path_name: str = (
        f"uu_ii={USER_LAYERS}_{ITEM_LAYERS}"
        f"_{USER_LOSS_RATIO}_{ITEM_LOSS_RATIO}"
        f"_topk={TOPK}_t={TEMPERATURE}"
        f"_regs={REGS}_dim={EMBED_SIZE}"
        f"_seed={seed}_{variant_name}"
    )
    log_file: str = f"../{DATASET}/{path_name}/{path_name}.txt"
    if not os.path.exists(log_file):
        print(f"  [warn] log file missing: {log_file}")
        return None
    with open(log_file, encoding="utf-8", errors="replace") as fh:
        text: str = fh.read()
    out: dict[str, float] = {}
    for metric, pat in _BEST_PATTERNS.items():
        m = pat.search(text)
        if not m:
            m = _FALLBACK_PATTERNS[metric].search(text)
        if m:
            out[metric] = float(m.group(1))
    if len(out) != len(_BEST_PATTERNS):
        print(f"  [warn] partial metrics parsed from {log_file}: {out}")
        return None
    out["log_file"] = log_file
    return out


# -- Run loop ------------------------------------------------------------
env: dict[str, str] = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUNBUFFERED"] = "1"

for variant_idx, variant_name in enumerate(ABLATION_VARIANTS, start=1):
    variant = get_variant(variant_name)
    for seed_idx, seed in enumerate(ABLATION_SEEDS, start=1):
        run_id: str = f"{variant_name}/seed={seed}"
        print(f"\n{'#' * 80}")
        print(
            f"# [{variant_idx}/{len(ABLATION_VARIANTS)}] "
            f"variant={variant_name}  seed={seed}  "
            f"({seed_idx}/{len(ABLATION_SEEDS)})"
        )
        print(f"# notes: {variant.notes}")
        print(f"{'#' * 80}")

        cmd: list[str] = _build_run_cmd(variant_name, seed)
        t0: float = time.time()
        result: subprocess.CompletedProcess[str] = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            env=env,
            encoding="utf-8",
            errors="replace",
        )
        dt: float = time.time() - t0

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"\n[WARNING] {run_id} exited with code {result.returncode}")
            if result.stderr:
                print("[STDERR]:")
                print(result.stderr)

        metrics = _parse_log(variant_name, seed)
        row: dict[str, Any] = {
            "variant": variant_name,
            "seed": seed,
            "notes": variant.notes,
            "duration_sec": dt,
            "return_code": result.returncode,
        }
        if metrics is not None:
            row.update(metrics)
            print(
                f"  metrics: recall@20={metrics['recall@20']:.6f}  "
                f"precision@20={metrics['precision@20']:.6f}  "
                f"ndcg@20={metrics['ndcg@20']:.6f}  "
                f"(elapsed {dt / 60:.1f} min)"
            )
        else:
            row["recall@20"] = float("nan")
            row["precision@20"] = float("nan")
            row["ndcg@20"] = float("nan")
            print(f"  metrics: UNAVAILABLE  (elapsed {dt / 60:.1f} min)")
        ABLATION_RESULTS.append(row)

print(f"\n{'=' * 80}")
print(
    f"ALL ABLATION RUNS COMPLETED: {len(ABLATION_RESULTS)} rows "
    f"({len(ABLATION_VARIANTS)} variants x {len(ABLATION_SEEDS)} seeds)"
)
print(f"{'=' * 80}")

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd

# -- Build a long-format dataframe ---------------------------------------
if not ABLATION_RESULTS:
    raise RuntimeError("ABLATION_RESULTS is empty: run the previous cell first.")

df_long: pd.DataFrame = pd.DataFrame(ABLATION_RESULTS)
print("RAW ABLATION TABLE (long-format)")
print("-" * 80)
print(df_long.to_string(index=False))

# -- Mean ± std per variant ----------------------------------------------
METRIC_COLS: list[str] = ["recall@20", "precision@20", "ndcg@20"]
mean_df: pd.DataFrame = (
    df_long.groupby("variant", sort=False)[METRIC_COLS].mean().round(6)
)
std_df: pd.DataFrame = (
    df_long.groupby("variant", sort=False)[METRIC_COLS].std(ddof=1).round(6)
)
mean_df.columns = [f"mean_{c}" for c in mean_df.columns]
std_df.columns = [f"std_{c}" for c in std_df.columns]
summary: pd.DataFrame = pd.concat([mean_df, std_df], axis=1)

# -- Marginal delta (metric − metric(A0_full)) ----------------------------
if "A0_full" not in summary.index:
    print("\n[warn] A0_full not found in summary; delta columns will be NaN.")
    baseline = {m: float("nan") for m in METRIC_COLS}
else:
    baseline = {m: summary.loc["A0_full", f"mean_{m}"] for m in METRIC_COLS}

for m in METRIC_COLS:
    summary[f"delta_{m}"] = summary[f"mean_{m}"] - baseline[m]

summary = summary.reset_index().rename(columns={"variant": "variant_id"})
summary["notes"] = summary["variant_id"].map(
    {v: get_variant(v).notes for v in ABLATION_VARIANTS}
)

# Reorder columns for readability: variant, notes, means, stds, deltas.
col_order: list[str] = (
    ["variant_id", "notes"]
    + [f"mean_{m}" for m in METRIC_COLS]
    + [f"std_{m}" for m in METRIC_COLS]
    + [f"delta_{m}" for m in METRIC_COLS]
)
summary = summary[col_order]

print(f"\n{'=' * 80}")
print("ABLATION SUMMARY (mean ± std + marginal delta vs A0_full)")
print(f"{'=' * 80}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print(summary.to_string(index=False))

# -- Persist to CSV ------------------------------------------------------
OUT_DIR: str = os.path.join(PROJECT_ROOT, "log")
os.makedirs(OUT_DIR, exist_ok=True)
raw_csv: str = os.path.join(OUT_DIR, "ablation_raw.csv")
summary_csv: str = os.path.join(OUT_DIR, "ablation_summary.csv")
df_long.to_csv(raw_csv, index=False)
summary.to_csv(summary_csv, index=False)
print(f"\nSaved:\n  {raw_csv}\n  {summary_csv}")

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon

# -- Build seed-aligned pivot tables: rows = seeds, cols = variants -------
METRIC_COLS: list[str] = ["recall@20", "precision@20", "ndcg@20"]
pivot_tables: dict[str, pd.DataFrame] = {
    m: df_long.pivot(index="seed", columns="variant", values=m)[ABLATION_VARIANTS]
    for m in METRIC_COLS
}

stat_rows: list[dict] = []
for metric in METRIC_COLS:
    pivot = pivot_tables[metric]
    if "A0_full" not in pivot.columns:
        print(f"[warn] A0_full missing; skipping stats for {metric}")
        continue
    base = pivot["A0_full"].dropna()
    if len(base) < 2:
        print(f"[warn] insufficient A0_full seeds ({len(base)}); skipping {metric}")
        continue

    for variant in pivot.columns:
        if variant == "A0_full":
            continue
        other = pivot[variant].reindex(base.index).dropna()
        common = base.index.intersection(other.index)
        if len(common) < 2:
            stat_rows.append({
                "variant_id": variant,
                "metric": metric,
                "n_seeds": len(common),
                "mean_A0_full": float("nan"),
                "mean_variant": float("nan"),
                "delta": float("nan"),
                "ttest_p": float("nan"),
                "wilcoxon_p": float("nan"),
            })
            continue
        a = base.loc[common].to_numpy()
        b = other.loc[common].to_numpy()
        delta = float(b.mean() - a.mean())
        try:
            t_stat, t_p = ttest_rel(a, b)
        except Exception:
            t_stat, t_p = float("nan"), float("nan")
        try:
            # Wilcoxon fails if all differences are zero; guard against that.
            if np.allclose(a, b):
                w_p = 1.0
            else:
                _w, w_p = wilcoxon(a, b, zero_method="wilcox", mode="auto")
        except Exception:
            w_p = float("nan")
        stat_rows.append({
            "variant_id": variant,
            "metric": metric,
            "n_seeds": int(len(common)),
            "mean_A0_full": float(a.mean()),
            "mean_variant": float(b.mean()),
            "delta": delta,
            "ttest_p": float(t_p),
            "wilcoxon_p": float(w_p),
        })

stats_df: pd.DataFrame = pd.DataFrame(stat_rows)
if not stats_df.empty:
    stats_df = stats_df.sort_values(["metric", "variant_id"]).reset_index(drop=True)

print("PAIRED STATISTICAL TESTS vs A0_full")
print("=" * 80)
if stats_df.empty:
    print("(no paired tests produced -- insufficient seeds or missing A0_full)")
else:
    print(stats_df.to_string(index=False))

# Persist.
stats_csv: str = os.path.join(OUT_DIR, "ablation_paired_stats.csv")
stats_df.to_csv(stats_csv, index=False)
print(f"\nSaved: {stats_csv}")

In [ ]:
from __future__ import annotations

import os

import pandas as pd

try:
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
    _HAS_OPENPYXL: bool = True
except ImportError:
    _HAS_OPENPYXL = False

OUT_DIR: str = os.path.join(PROJECT_ROOT, "log")
os.makedirs(OUT_DIR, exist_ok=True)
excel_path: str = os.path.join(OUT_DIR, "ablation_results.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_long.to_excel(writer, sheet_name="raw", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    if not stats_df.empty:
        stats_df.to_excel(writer, sheet_name="paired_stats", index=False)

    # Per-metric pivot sheets (rows = seeds, cols = variants)
    for metric in METRIC_COLS:
        sheet = f"pivot_{metric.replace('@', '_')}"
        pivot_tables[metric].to_excel(writer, sheet_name=sheet)

    if _HAS_OPENPYXL:
        workbook = writer.book
        header_font = Font(bold=True, color="FFFFFF")
        header_fill = PatternFill("solid", fgColor="305496")
        centre = Alignment(horizontal="center", vertical="center")
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            for cell in sheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = centre
            for col_idx, column in enumerate(sheet.columns, start=1):
                max_len = max(
                    (len(str(c.value)) for c in column if c.value is not None),
                    default=12,
                )
                sheet.column_dimensions[get_column_letter(col_idx)].width = min(
                    max(max_len + 2, 10), 48
                )

print(f"Exported ablation workbook: {excel_path}")
print("Sheets:")
with pd.ExcelFile(excel_path) as xf:
    for s in xf.sheet_names:
        print(f"  - {s}")